# 第三部分：RAII 与资源生命周期

## 实验 3：作用域就是资源生命周期

实验 2 已经把文件句柄绑定到对象。本实验继续回答一个更具体的问题：RAII 对象究竟在什么时候析构？

对于局部对象，最重要的答案是：**控制流离开对象所在的作用域时。** 因此可以通过设计作用域，精确表达资源需要保持多久。

本实验将观察：

- 内层作用域结束时立即释放资源；
- 同一作用域中的对象按构造顺序的逆序析构；
- 外层资源可以跨越多个内层作用域继续存活；
- 每次循环迭代都有独立的局部对象生命周期；
- 对象拥有的资源成员会随拥有者一起销毁。

所有文件输出都位于 `outputs/03/`。

In [1]:
#include <cstdio>
#include <filesystem>
#include <iostream>
#include <stdexcept>
#include <string>

### 1. 定义可观察的文件资源

`ScopedFile` 延续实验 2 的唯一所有权模型，并增加 `label_`，让构造和析构顺序在输出中清晰可见。

In [2]:
std::filesystem::create_directories("outputs/03");

class ScopedFile
{
public:
    ScopedFile(
        const std::string &label,
        const std::string &path,
        const char *mode)
        : label_(label),
          path_(path),
          file_(std::fopen(path_.c_str(), mode))
    {
        if (file_ == nullptr)
        {
            throw std::runtime_error(
                std::string("cannot open file: ") + path_);
        }

        std::cout << label_ << " acquire" << std::endl;
    }

    ~ScopedFile()
    {
        std::cout << label_ << " release" << std::endl;
        std::fclose(file_);
    }

    ScopedFile(const ScopedFile &) = delete;
    ScopedFile &operator=(const ScopedFile &) = delete;

    std::FILE *get() const
    {
        return file_;
    }

private:
    std::string label_;
    std::string path_;
    std::FILE *file_;
};

### 2. 用 `A–E` 定位析构时刻

`file` 只属于内层 `{}`。执行内层右花括号时，它的生命周期立即结束。

In [3]:
{
    std::cout << "A" << std::endl;

    {
        std::cout << "B" << std::endl;

        ScopedFile file(
            "inner file",
            "outputs/03/output.txt",
            "w");

        std::cout << "C" << std::endl;
        std::fputs("Scoped Resource", file.get());
        std::cout << "D" << std::endl;
    }

    std::cout << "E" << std::endl;
}

A
B
inner file acquire
C
D
inner file release
E


预期顺序：

```text
A
B
inner file acquire
C
D
inner file release
E
```

`release` 出现在 `D` 和 `E` 之间，说明析构发生在离开内层作用域时。资源不必存活到外层作用域或程序结束。

### 3. 同一作用域按构造顺序的逆序析构

局部对象按照声明执行到的位置依次构造；离开共同作用域时，后构造的对象先析构。

In [4]:
{
    ScopedFile first(
        "first",
        "outputs/03/first.txt",
        "w");

    ScopedFile second(
        "second",
        "outputs/03/second.txt",
        "w");

    std::cout << "use both files" << std::endl;
}

first acquire
second acquire
use both files
second release
first release


预期输出中的资源顺序是：

```text
first acquire
second acquire
second release
first release
```

这种后进先出的清理顺序非常重要：后创建的资源通常依赖先创建的资源，因此应该先被释放。

### 4. 外层资源跨越内层作用域

对象只在自己的作用域结束时析构。内层资源被释放后，外层资源仍然有效。

In [5]:
{
    ScopedFile outer(
        "outer",
        "outputs/03/outer.txt",
        "w");

    std::fputs("before inner scope", outer.get());

    {
        ScopedFile inner(
            "inner",
            "outputs/03/inner.txt",
            "w");
        std::fputs("inside inner scope", inner.get());
    }

    std::fputs("after inner scope", outer.get());
    std::cout << "outer is still usable" << std::endl;
}

outer acquire
inner acquire
inner release
outer is still usable
outer release


### 5. 主动缩小作用域

不必让资源对象一直活到整个函数结束。额外的 `{}` 可以表达“只在这段操作中需要资源”：

```cpp
prepare();
{
    ScopedFile temporary(...);
    write(temporary);
} // 在这里立即释放，而不是等函数结束
continue_work();
```

缩短生命周期可以减少同时占用的稀缺资源，也能缩小借用指针或锁的有效范围。实际工程中应让资源作用域尽可能小且清晰。

### 6. 每次循环迭代都有独立生命周期

循环体本身也是作用域。每轮创建的局部资源会在本轮末尾释放，然后下一轮再创建新资源。

In [6]:
for (int index = 1; index <= 3; ++index)
{
    const std::string label =
        "iteration " + std::to_string(index);
    const std::string path =
        "outputs/03/iteration-"
        + std::to_string(index)
        + ".txt";

    ScopedFile file(label, path, "w");
    std::fputs("one iteration", file.get());
}

iteration 1 acquire
iteration 1 release
iteration 2 acquire
iteration 2 release
iteration 3 acquire
iteration 3 release


输出应当交替出现 `iteration N acquire` 和 `iteration N release`，而不是先获取三个资源、最后统一释放。这能限制批处理任务同时持有的文件或连接数量。

### 7. 资源成员随拥有者一起销毁

RAII 对象还可以成为另一个类的成员。拥有者构造时依次构造成员；拥有者析构函数体执行完后，成员按声明顺序的逆序析构。

In [7]:
class ReportFiles
{
public:
    ReportFiles()
        : data_(
              "data member",
              "outputs/03/report-data.txt",
              "w"),
          log_(
              "log member",
              "outputs/03/report-log.txt",
              "w")
    {
        std::cout << "ReportFiles ready" << std::endl;
    }

    ~ReportFiles()
    {
        std::cout << "ReportFiles destructor body" << std::endl;
    }

private:
    ScopedFile data_;
    ScopedFile log_;
};

{
    ReportFiles report;
}

data member acquire
log member acquire
ReportFiles ready
ReportFiles destructor body
log member release
data member release


关键顺序是：

```text
data member acquire
log member acquire
ReportFiles ready
ReportFiles destructor body
log member release
data member release
```

成员的实际构造顺序由它们在类中的声明顺序决定，而不是初始化列表的书写顺序；析构顺序则与声明顺序相反。

### 8. 更准确的说法是“作用域对象”

局部对象常被口语化称为“栈对象”，但本实验真正依赖的是 C++ 的自动存储期和作用域规则，而不是某个具体的栈内存实现。看到局部 RAII 对象时，应优先思考它在哪个作用域构造、控制流何时离开该作用域。

### 9. RAII 可以管理什么

RAII 并不是智能指针的同义词。几乎所有具有“获取—使用—释放”协议的资源都可以绑定到对象生命周期：

| Resource | Acquire | Release |
| --- | --- | --- |
| Heap memory | `malloc/new` | `free/delete` |
| File | `fopen` | `fclose` |
| Socket | `socket` | `close` |
| Mutex | `lock` | `unlock` |
| Thread | create | `join/detach` |
| Database connection | connect | disconnect |
| GPU buffer | allocate | free |
| Native handle | acquire | matching close API |

例如 `std::lock_guard` 在构造时加锁、析构时解锁。额外的 `{}` 能精确限定临界区，原理与本实验中的 `ScopedFile` 完全相同。

### 实验结论

RAII 不只是“最终会自动释放”，还允许代码用词法作用域精确描述资源的存活区间：进入作用域后资源有效，离开作用域时按依赖关系逆序释放。

因此，设计资源管理代码时应让拥有者靠近使用位置、让作用域尽可能小，并利用成员对象组合多个资源，而不是把释放操作散落在控制流中。